# Batch single-FOV FLIM fitting – Auto‑global threshold with area filtering

Fits every `.ptu` in a folder with a 3‑exponential reconvolution model.  
The mask is generated per file using:
1. Pre‑clipping to a high percentile (removes outliers).
2. A global threshold method (Otsu, Yen, or Triangle).
3. Morphological cleanup (opening/closing).
4. Area filtering to keep only large cell‑sized components.

This approach is robust across files with different brightness levels.

**Settings** – edit the config cell below, then `Run All`.

| Parameter | Value |
|-----------|-------|
| Exponentials | 3 |
| τ range | 0.145 – 45 ns |
| IRF | Machine IRF (peak‑aligned to decay) |
| Optimizer | Differential evolution → LM polish |
| Mask method | auto_global (per‑file threshold) |
| Output | `<PTU_FOLDER>/batch_fit_results_auto_global.csv` |

In [7]:
from pathlib import Path

# ── Edit these ────────────────────────────────────────────────────────────────
PTU_FOLDER       = Path("/Volumes/Lexar/Layla/260309LMCR196TCAFADO40x.sptw/")
OUTPUT_CSV       = PTU_FOLDER / "batch_fit_results_auto_global.csv"

# Fitting parameters
N_EXP            = 3
TAU_MIN_NS       = 0.145        # ns
TAU_MAX_NS       = 45.0         # ns

# Machine IRF — leave None to use FLIMKit default
MACHINE_IRF_PATH = None

# DE optimiser
DE_POPSIZE       = 15
DE_MAXITER       = 1000
WORKERS          = -1           # -1 = all cores

# ── Masking parameters (auto_global) ─────────────────────────────────────────
#   'auto_global' – per‑file global threshold (recommended)
#   'none'        – use all photons (fastest, but includes background)
#   'threshold'   – global threshold with a fixed value (simple)
#   'adaptive'    – adaptive threshold + morphological cleanup (old)
MASK_MODE          = 'auto_global'

# Auto‑global parameters (used only if MASK_MODE == 'auto_global')
AUTO_GLOBAL_METHOD = 'triangle'   # 'otsu', 'yen', or 'triangle'
PRE_CLIP_PERCENTILE = 99.5        # clip intensity to this percentile before thresholding
MORPH_RADIUS        = 1           # radius of disk for opening/closing
MIN_AREA            = 5000        # keep only components larger than this (pixels)
KEEP_LARGEST        = False       # if True, overrides MIN_AREA (keeps only the largest)

# Global threshold (used if MASK_MODE == 'threshold')
INTENSITY_THRESHOLD = 5           # photons/pixel

# Adaptive parameters (used if MASK_MODE == 'adaptive')
ADAPTIVE_BLOCK_SIZE = 51
ADAPTIVE_OFFSET     = 2

# Minimum total photons per file to attempt fitting
MIN_PHOTONS        = 1_000

# ── Visualisation parameters ──────────────────────────────────────────────────
VISUAL_CLIP_PERCENTILE = 99       # clip intensity images at this percentile (0‑100)
CONTOUR_COLOR          = 'cyan'   # colour of mask outline
CONTOUR_LINEWIDTH      = 0.5
COLORMAP               = 'hot'    # colormap for saved intensity images

# ──────────────────────────────────────────────────────────────────────────────

In [8]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from flimkit.PTU.reader      import PTUFile
from flimkit.FLIM.fitters    import fit_summed
from flimkit.configs         import (
    MACHINE_IRF_DEFAULT_PATH,
    MACHINE_IRF_FIT_BG, MACHINE_IRF_FIT_SIGMA, MACHINE_IRF_FIT_TAIL,
)

# For thresholding and morphology
from skimage.filters import threshold_otsu, threshold_yen, threshold_triangle, threshold_local
from skimage.morphology import disk, binary_opening, binary_closing
from skimage import measure

# Resolve IRF
_irf_path = Path(MACHINE_IRF_PATH) if MACHINE_IRF_PATH else Path(MACHINE_IRF_DEFAULT_PATH)
if not _irf_path.exists():
    raise FileNotFoundError(f"Machine IRF not found: {_irf_path}")

_machine_irf_raw = np.load(str(_irf_path)).ravel().astype(float)
_machine_irf_raw = np.maximum(_machine_irf_raw, 0.0)
_machine_irf_raw /= _machine_irf_raw.sum()
PI_MACHINE = int(np.argmax(_machine_irf_raw))

print(f"Machine IRF : {_irf_path.name}  |  {len(_machine_irf_raw)} bins  |  peak bin {PI_MACHINE}")
print(f"FIT_BG={MACHINE_IRF_FIT_BG}  FIT_SIGMA={MACHINE_IRF_FIT_SIGMA}  HAS_TAIL={MACHINE_IRF_FIT_TAIL}")
print(f"n_exp={N_EXP}  τ=[{TAU_MIN_NS}, {TAU_MAX_NS}] ns")
print(f"Mask mode   : {MASK_MODE!r}")
if MASK_MODE == 'auto_global':
    print(f"  method={AUTO_GLOBAL_METHOD}, clip={PRE_CLIP_PERCENTILE}%, morph_radius={MORPH_RADIUS}, min_area={MIN_AREA}, keep_largest={KEEP_LARGEST}")
elif MASK_MODE == 'threshold':
    print(f"  threshold={INTENSITY_THRESHOLD} ph/px")
elif MASK_MODE == 'adaptive':
    print(f"  block_size={ADAPTIVE_BLOCK_SIZE}, offset={ADAPTIVE_OFFSET}, morph_radius={MORPH_RADIUS}")
print(f"Visualisation: clip to {VISUAL_CLIP_PERCENTILE}th percentile, colormap={COLORMAP}, contour={CONTOUR_COLOR}")

def get_irf(peak_bin: int, n_bins: int) -> np.ndarray:
    irf = _machine_irf_raw.copy()
    if irf.size > n_bins:
        irf = irf[:n_bins]
    elif irf.size < n_bins:
        irf = np.pad(irf, (0, n_bins - irf.size))
    shift = peak_bin - PI_MACHINE
    if shift:
        irf = np.roll(irf, shift)
    s = irf.sum()
    return irf / s if s > 0 else irf

def build_masked_decay(ptu: PTUFile) -> tuple[np.ndarray, int, str]:
    """
    Return (masked_decay, n_photons_used, mask_description).
    Applies MASK_MODE to restrict which pixels contribute to the summed decay.
    """
    if MASK_MODE == 'none':
        decay = ptu.summed_decay()
        return decay, int(decay.sum()), 'all pixels'

    # All masking modes need the intensity image
    stack = ptu.raw_pixel_stack(channel=ptu.photon_channel)  # (Y, X, H)
    intensity = stack.sum(axis=-1)                            # (Y, X)

    if MASK_MODE == 'threshold':
        mask = intensity >= INTENSITY_THRESHOLD
        desc = f"threshold≥{INTENSITY_THRESHOLD}"

    elif MASK_MODE == 'adaptive':
        thresh = threshold_local(intensity, ADAPTIVE_BLOCK_SIZE, method='gaussian', offset=ADAPTIVE_OFFSET)
        mask = intensity > thresh
        selem = disk(MORPH_RADIUS)
        mask = binary_opening(mask, selem)
        mask = binary_closing(mask, selem)
        desc = f"adaptive (block={ADAPTIVE_BLOCK_SIZE}, offset={ADAPTIVE_OFFSET}, r={MORPH_RADIUS})"

    elif MASK_MODE == 'auto_global':
        # 1. Pre‑clip to remove extreme outliers
        cap_val = np.percentile(intensity, PRE_CLIP_PERCENTILE)
        intensity_clipped = np.clip(intensity, 0, cap_val)

        # 2. Global threshold
        if AUTO_GLOBAL_METHOD == 'otsu':
            thresh = threshold_otsu(intensity_clipped)
        elif AUTO_GLOBAL_METHOD == 'yen':
            thresh = threshold_yen(intensity_clipped)
        elif AUTO_GLOBAL_METHOD == 'triangle':
            thresh = threshold_triangle(intensity_clipped)
        else:
            raise ValueError(f"Unknown AUTO_GLOBAL_METHOD: {AUTO_GLOBAL_METHOD}")
        mask = intensity_clipped >= thresh

        # 3. Morphological cleanup
        if MORPH_RADIUS > 0:
            selem = disk(MORPH_RADIUS)
            mask = binary_opening(mask, selem)
            mask = binary_closing(mask, selem)

        # 4. Area filtering (keep only components larger than MIN_AREA)
        if MIN_AREA > 0 and not KEEP_LARGEST:
            labeled = measure.label(mask)
            areas = np.bincount(labeled.ravel())[1:]  # areas of each component
            keep_labels = np.where(areas >= MIN_AREA)[0] + 1
            mask = np.isin(labeled, keep_labels)

        # 5. Optionally keep only the largest component
        if KEEP_LARGEST:
            labeled = measure.label(mask)
            if labeled.max() > 0:
                largest_label = np.argmax(np.bincount(labeled.ravel())[1:]) + 1
                mask = (labeled == largest_label)

        desc = (f"auto_{AUTO_GLOBAL_METHOD} (clip={PRE_CLIP_PERCENTILE}%, "
                f"r={MORPH_RADIUS}, min_area={MIN_AREA})")
        if KEEP_LARGEST:
            desc += " + keep_largest"

    else:
        raise ValueError(f"Unknown MASK_MODE: {MASK_MODE!r}. "
                         "Choose 'none', 'threshold', 'adaptive', or 'auto_global'.")

    # Apply mask: keep only pixels where mask is True
    stack[~mask] = 0
    decay = stack.sum(axis=(0, 1))
    n_ph  = int(decay.sum())
    del stack
    return decay, n_ph, desc

print("\nImports OK ✓")

Machine IRF : machine_irf_default.npy  |  526 bins  |  peak bin 29
FIT_BG=True  FIT_SIGMA=False  HAS_TAIL=False
n_exp=3  τ=[0.145, 45.0] ns
Mask mode   : 'auto_global'
  method=triangle, clip=99.5%, morph_radius=1, min_area=5000, keep_largest=False
Visualisation: clip to 99th percentile, colormap=hot, contour=cyan

Imports OK ✓


In [9]:
ptu_files = sorted(PTU_FOLDER.glob("*.ptu"))
print(f"Found {len(ptu_files)} .ptu files in {PTU_FOLDER}")
for p in ptu_files[:5]:
    print(f"  {p.name}")
if len(ptu_files) > 5:
    print(f"  ... and {len(ptu_files)-5} more")

Found 32 .ptu files in /Volumes/Lexar/Layla/260309LMCR196TCAFADO40x.sptw
  AB680_1.ptu
  AB680_2.ptu
  AB680_Ado_1.ptu
  AB680_Ado_2.ptu
  Ado_1.ptu
  ... and 27 more


In [10]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

records = []
errors  = []

for ptu_path in tqdm(ptu_files, desc="Fitting"):
    row = {"file": ptu_path.name}
    try:
        ptu    = PTUFile(str(ptu_path), verbose=False)
        _      = ptu.summed_decay()                  # sets ptu.photon_channel
        n_bins = ptu.n_bins
        tcspc  = ptu.tcspc_res

        row["n_bins"]   = n_bins
        row["tcspc_ps"] = round(tcspc * 1e12, 4)

        # Build mask and masked decay
        decay, n_ph, mask_desc = build_masked_decay(ptu)
        row["n_photons"]  = n_ph
        row["mask_mode"]  = mask_desc
        row["peak_bin"]   = int(np.argmax(decay))

        # ── Save intensity image with mask overlay (clipped to percentile) ────
        # Recompute intensity and mask for visualisation (to avoid altering the decay)
        intensity = ptu.raw_pixel_stack(channel=ptu.photon_channel).sum(axis=-1)
        if MASK_MODE == 'none':
            intensity_masked = intensity
            img_filename = ptu_path.stem + "_intensity_unmasked.png"
        else:
            # Recompute the mask using the same logic (for visualisation only)
            if MASK_MODE == 'threshold':
                mask = intensity >= INTENSITY_THRESHOLD
                img_filename = ptu_path.stem + f"_intensity_threshold_{INTENSITY_THRESHOLD}.png"
            elif MASK_MODE == 'adaptive':
                thresh = threshold_local(intensity, ADAPTIVE_BLOCK_SIZE, method='gaussian', offset=ADAPTIVE_OFFSET)
                mask = intensity > thresh
                selem = disk(MORPH_RADIUS)
                mask = binary_opening(mask, selem)
                mask = binary_closing(mask, selem)
                img_filename = ptu_path.stem + f"_intensity_adaptive_{ADAPTIVE_BLOCK_SIZE}_{ADAPTIVE_OFFSET}_r{MORPH_RADIUS}.png"
            elif MASK_MODE == 'auto_global':
                # Recreate the exact mask used for fitting
                cap_val = np.percentile(intensity, PRE_CLIP_PERCENTILE)
                intensity_clipped = np.clip(intensity, 0, cap_val)
                if AUTO_GLOBAL_METHOD == 'otsu':
                    thresh = threshold_otsu(intensity_clipped)
                elif AUTO_GLOBAL_METHOD == 'yen':
                    thresh = threshold_yen(intensity_clipped)
                elif AUTO_GLOBAL_METHOD == 'triangle':
                    thresh = threshold_triangle(intensity_clipped)
                mask = intensity_clipped >= thresh
                if MORPH_RADIUS > 0:
                    selem = disk(MORPH_RADIUS)
                    mask = binary_opening(mask, selem)
                    mask = binary_closing(mask, selem)
                if MIN_AREA > 0 and not KEEP_LARGEST:
                    labeled = measure.label(mask)
                    areas = np.bincount(labeled.ravel())[1:]
                    keep_labels = np.where(areas >= MIN_AREA)[0] + 1
                    mask = np.isin(labeled, keep_labels)
                if KEEP_LARGEST:
                    labeled = measure.label(mask)
                    if labeled.max() > 0:
                        largest_label = np.argmax(np.bincount(labeled.ravel())[1:]) + 1
                        mask = (labeled == largest_label)
                img_filename = ptu_path.stem + f"_intensity_auto_{AUTO_GLOBAL_METHOD}_clip{PRE_CLIP_PERCENTILE}_r{MORPH_RADIUS}_area{MIN_AREA}.png"

            intensity_masked = intensity.copy()
            intensity_masked[~mask] = 0

        # Clip for visualisation (99th percentile of non‑zero pixels)
        nonzero = intensity_masked[intensity_masked > 0]
        if len(nonzero) > 0:
            vmax = np.percentile(nonzero, VISUAL_CLIP_PERCENTILE)
        else:
            vmax = 1

        fig, ax = plt.subplots(figsize=(8, 8))
        im = ax.imshow(intensity_masked, cmap=COLORMAP, vmin=0, vmax=vmax, origin="upper")
        # Overlay mask contour if a mask exists and has any pixels
        if MASK_MODE != 'none' and mask.sum() > 0:
            ax.contour(mask, levels=[0.5], colors=CONTOUR_COLOR, linewidths=CONTOUR_LINEWIDTH)
        ax.set_title(f"{ptu_path.name}\nMask: {mask_desc} | Photons retained: {n_ph:,}")
        plt.colorbar(im, ax=ax, label="Photons/pixel")
        plt.tight_layout()
        plt.savefig(PTU_FOLDER / img_filename, dpi=100, bbox_inches="tight")
        plt.close()

        if n_ph < MIN_PHOTONS:
            row["status"] = f"SKIPPED (<{MIN_PHOTONS} photons)"
            records.append(row)
            continue

        irf = get_irf(row["peak_bin"], n_bins)

        popt, s = fit_summed(
            decay, tcspc, n_bins, irf,
            has_tail      = MACHINE_IRF_FIT_TAIL,
            fit_bg        = MACHINE_IRF_FIT_BG,
            fit_sigma     = MACHINE_IRF_FIT_SIGMA,
            n_exp         = N_EXP,
            tau_min_ns    = TAU_MIN_NS,
            tau_max_ns    = TAU_MAX_NS,
            optimizer     = "de",
            de_popsize    = DE_POPSIZE,
            de_maxiter    = DE_MAXITER,
            workers       = WORKERS,
            polish        = True,
        )

        taus  = s["taus_ns"]
        amps  = s["amps"]
        fracs = s["fractions"]

        for k in range(N_EXP):
            row[f"tau{k+1}_ns"] = round(float(taus[k]),  5)
            row[f"amp{k+1}"]    = round(float(amps[k]),  4)
            row[f"frac{k+1}"]   = round(float(fracs[k]), 5)

        row["tau_mean_amp_ns"]   = round(s["tau_mean_amp_ns"],   5)
        row["tau_mean_int_ns"]   = round(s["tau_mean_int_ns"],   5)
        row["bg_fit"]            = round(float(s["bg_fit"]),     6)
        row["chi2_reduced"]      = round(s["reduced_chi2"],      5)
        row["chi2_reduced_tail"] = round(s["reduced_chi2_tail"], 5)
        row["chi2_full"]         = round(s["chi2"],              4)
        row["dof"]               = int(s["dof"])
        row["p_val"]             = round(s["p_val"],             6)
        row["irf_shift_bins"]    = round(float(s["irf_shift_bins"]), 4)
        row["irf_fwhm_eff_ns"]   = round(float(s["irf_fwhm_eff_ns"]), 4)
        row["fit_window_ns_lo"]  = round(s["fit_window_ns"][0], 4)
        row["fit_window_ns_hi"]  = round(s["fit_window_ns"][1], 4)
        row["optimizer_msg"]     = str(s["optimizer_msg"])[:80]
        row["intensity_image"]   = img_filename
        row["status"]            = "OK"

    except Exception as e:
        row["status"] = f"ERROR: {e}"
        errors.append(ptu_path.name)
        import traceback; traceback.print_exc()

    records.append(row)
    # ── Save intensity image with mask overlay ────
    # Recompute intensity and mask for visualisation (to avoid altering the decay)
    intensity = ptu.raw_pixel_stack(channel=ptu.photon_channel).sum(axis=-1)

    if MASK_MODE == 'none':
        intensity_masked = intensity
        img_filename = ptu_path.stem + "_intensity_unmasked.png"
    else:
        # Recompute the mask using the same logic (for visualisation only)
        if MASK_MODE == 'threshold':
            mask = intensity >= INTENSITY_THRESHOLD
            img_filename = ptu_path.stem + f"_intensity_threshold_{INTENSITY_THRESHOLD}.png"
        elif MASK_MODE == 'adaptive':
            thresh = threshold_local(intensity, ADAPTIVE_BLOCK_SIZE, method='gaussian', offset=ADAPTIVE_OFFSET)
            mask = intensity > thresh
            selem = disk(MORPH_RADIUS)
            mask = binary_opening(mask, selem)
            mask = binary_closing(mask, selem)
            img_filename = ptu_path.stem + f"_intensity_adaptive_{ADAPTIVE_BLOCK_SIZE}_{ADAPTIVE_OFFSET}_r{MORPH_RADIUS}.png"
        elif MASK_MODE == 'auto_global':
            # Recreate the exact mask used for fitting
            cap_val = np.percentile(intensity, PRE_CLIP_PERCENTILE)
            intensity_clipped = np.clip(intensity, 0, cap_val)
            if AUTO_GLOBAL_METHOD == 'otsu':
                thresh = threshold_otsu(intensity_clipped)
            elif AUTO_GLOBAL_METHOD == 'yen':
                thresh = threshold_yen(intensity_clipped)
            elif AUTO_GLOBAL_METHOD == 'triangle':
                thresh = threshold_triangle(intensity_clipped)
            mask = intensity_clipped >= thresh
            if MORPH_RADIUS > 0:
                selem = disk(MORPH_RADIUS)
                mask = binary_opening(mask, selem)
                mask = binary_closing(mask, selem)
            if MIN_AREA > 0 and not KEEP_LARGEST:
                labeled = measure.label(mask)
                areas = np.bincount(labeled.ravel())[1:]
                keep_labels = np.where(areas >= MIN_AREA)[0] + 1
                mask = np.isin(labeled, keep_labels)
            if KEEP_LARGEST:
                labeled = measure.label(mask)
                if labeled.max() > 0:
                    largest_label = np.argmax(np.bincount(labeled.ravel())[1:]) + 1
                    mask = (labeled == largest_label)
            img_filename = ptu_path.stem + f"_intensity_auto_{AUTO_GLOBAL_METHOD}_clip{PRE_CLIP_PERCENTILE}_r{MORPH_RADIUS}_area{MIN_AREA}.png"

        intensity_masked = intensity.copy()
        intensity_masked[~mask] = 0

    # Ensure the output directory exists
    PTU_FOLDER.mkdir(parents=True, exist_ok=True)

    # Clip for visualisation
    nonzero = intensity_masked[intensity_masked > 0]
    if len(nonzero) > 0:
        vmax = np.percentile(nonzero, VISUAL_CLIP_PERCENTILE)
    else:
        vmax = 1

    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(intensity_masked, cmap=COLORMAP, vmin=0, vmax=vmax, origin="upper")
    if MASK_MODE != 'none' and mask.sum() > 0:
        ax.contour(mask, levels=[0.5], colors=CONTOUR_COLOR, linewidths=CONTOUR_LINEWIDTH)
    ax.set_title(f"{ptu_path.name}\nMask: {mask_desc} | Photons retained: {n_ph:,}")
    plt.colorbar(im, ax=ax, label="Photons/pixel")
    plt.tight_layout()

    full_path = PTU_FOLDER / img_filename
    plt.savefig(full_path, dpi=100, bbox_inches="tight")
    plt.close()

    # Debug print
    print(f"Saved image: {full_path}")
print(f"\n{sum(1 for r in records if r['status']=='OK')}/{len(ptu_files)} fitted successfully")
if errors:
    print(f"Errors: {errors}")
print(f"Intensity images saved to {PTU_FOLDER}")

Fitting:   0%|          | 0/32 [00:00<?, ?it/s]

  Next-period artefact at bin 485 (47.03 ns). Truncating fit window.
  Cost function: poisson
  bg initial guess = 341.500 cts/bin, upper bound = 683.000 (free param)
  σ broadening: fixed at 0
  Fit window: bins 1–464 (0.10–44.99 ns), 463 bins
  Differential evolution: popsize=15, maxiter=1000, workers=-1
  Running final LM polish...
Saved image: /Volumes/Lexar/Layla/260309LMCR196TCAFADO40x.sptw/AB680_1_intensity_auto_triangle_clip99.5_r1_area5000.png
  Next-period artefact at bin 485 (47.03 ns). Truncating fit window.
  Cost function: poisson
  bg initial guess = 345.000 cts/bin, upper bound = 690.000 (free param)
  σ broadening: fixed at 0
  Fit window: bins 1–464 (0.10–44.99 ns), 463 bins
  Differential evolution: popsize=15, maxiter=1000, workers=-1
  Running final LM polish...
Saved image: /Volumes/Lexar/Layla/260309LMCR196TCAFADO40x.sptw/AB680_2_intensity_auto_triangle_clip99.5_r1_area5000.png
  Next-period artefact at bin 486 (47.13 ns). Truncating fit window.
  Cost function: 

In [11]:
df = pd.DataFrame(records)

front_cols = ["file", "status", "mask_mode", "n_photons", "tcspc_ps", "n_bins", "peak_bin"]
tau_cols   = []
for k in range(N_EXP):
    tau_cols += [f"tau{k+1}_ns", f"amp{k+1}", f"frac{k+1}"]
mean_cols  = ["tau_mean_amp_ns", "tau_mean_int_ns"]
fit_cols   = ["bg_fit", "chi2_reduced", "chi2_reduced_tail", "chi2_full",
              "dof", "p_val", "irf_shift_bins", "irf_fwhm_eff_ns",
              "fit_window_ns_lo", "fit_window_ns_hi", "optimizer_msg"]

ordered   = [c for c in front_cols + tau_cols + mean_cols + fit_cols if c in df.columns]
remaining = [c for c in df.columns if c not in ordered]
df = df[ordered + remaining]

df.to_csv(OUTPUT_CSV, index=False)
print(f"CSV saved → {OUTPUT_CSV}")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
df[df.status == "OK"].head(10)

CSV saved → /Volumes/Lexar/Layla/260309LMCR196TCAFADO40x.sptw/batch_fit_results_auto_global.csv
Shape: 32 rows × 30 columns



,file,status,mask_mode,n_photons,tcspc_ps,n_bins,peak_bin,tau1_ns,amp1,frac1,...,chi2_reduced_tail,chi2_full,dof,p_val,irf_shift_bins,irf_fwhm_eff_ns,fit_window_ns_lo,fit_window_ns_hi,optimizer_msg,intensity_image
0,AB680_1.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",3225738,96.9697,529,29,6.09099,16885.9474,0.07319,...,1.57214,4025.6653,455,0.0,-1.1500,0.097,0.097,44.9939,"DE success=True, fun=4.1022e+03; polished cost...",AB680_1_intensity_auto_triangle_clip99.5_r1_ar...
1,AB680_2.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",3772702,96.9697,530,29,6.14649,19700.6766,0.07338,...,1.62885,4645.1711,455,0.0,-1.1517,0.097,0.097,44.9939,"DE success=True, fun=4.7436e+03; polished cost...",AB680_2_intensity_auto_triangle_clip99.5_r1_ar...
2,AB680_Ado_1.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",2952811,96.9697,529,29,6.19039,15327.9830,0.07654,...,1.67894,3822.2376,455,0.0,-1.1483,0.097,0.097,44.9939,"DE success=True, fun=3.9054e+03; polished cost...",AB680_Ado_1_intensity_auto_triangle_clip99.5_r...
3,AB680_Ado_2.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",2629818,96.9697,531,29,6.26371,13416.7677,0.07407,...,1.51975,3620.7238,455,0.0,-1.1429,0.097,0.097,44.9939,"DE success=True, fun=3.6864e+03; polished cost...",AB680_Ado_2_intensity_auto_triangle_clip99.5_r...
4,Ado_1.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",1864407,96.9697,530,29,6.08315,10648.1887,0.07525,...,1.92055,2918.9354,455,0.0,-0.8434,0.097,0.097,44.9939,"DE success=True, fun=3.6408e+03; polished cost...",Ado_1_intensity_auto_triangle_clip99.5_r1_area...
5,Ado_2.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",5272864,96.9697,529,29,6.52925,24450.5311,0.06655,...,2.00279,10035.4197,455,0.0,-1.0572,0.097,0.097,44.9939,"DE success=True, fun=1.0197e+04; polished cost...",Ado_2_intensity_auto_triangle_clip99.5_r1_area...
6,Ctrl_1.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",5122125,96.9697,528,29,6.05799,28849.5477,0.07103,...,3.71101,4831.5464,455,0.0,-0.7260,0.097,0.097,44.9939,"DE success=True, fun=5.7411e+03; polished cost...",Ctrl_1_intensity_auto_triangle_clip99.5_r1_are...
7,Ctrl_2.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",4441938,96.9697,532,29,5.99278,25416.2904,0.07292,...,3.05269,5420.9329,455,0.0,-0.7724,0.097,0.097,44.9939,"DE success=True, fun=6.6727e+03; polished cost...",Ctrl_2_intensity_auto_triangle_clip99.5_r1_are...
8,Daratumab_1.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",3608362,96.9697,529,29,6.13886,18209.2444,0.07061,...,1.70399,4664.8016,455,0.0,-1.1496,0.097,0.097,44.9939,"DE success=True, fun=4.7696e+03; polished cost...",Daratumab_1_intensity_auto_triangle_clip99.5_r...
9,Daratumab_2.ptu,OK,"auto_triangle (clip=99.5%, r=1, min_area=5000)",3289598,96.9697,531,29,6.03308,17502.5681,0.07463,...,1.55016,4150.5321,455,0.0,-1.1448,0.097,0.097,44.9939,"DE success=True, fun=4.2420e+03; polished cost...",Daratumab_2_intensity_auto_triangle_clip99.5_r...


In [12]:
import matplotlib.pyplot as plt

ok = df[df.status == "OK"].copy()
if len(ok) == 0:
    print("No successful fits to plot.")
else:
    tau_keys = [f"tau{k+1}_ns" for k in range(N_EXP) if f"tau{k+1}_ns" in ok.columns]
    print("── Summary statistics ───────────────────────────────────────")
    cols_stat = tau_keys + ["tau_mean_amp_ns", "chi2_reduced_tail"]
    print(ok[cols_stat].describe().round(4).to_string())

    # Histogram of lifetimes
    fig, axes = plt.subplots(1, len(tau_keys) + 1, figsize=(4*(len(tau_keys)+1), 3.5))
    for ax, key in zip(axes, tau_keys):
        ax.hist(ok[key].dropna(), bins=20, color="#028090", edgecolor="white", alpha=0.85)
        ax.set_xlabel(key.replace("_", " "))
        ax.set_ylabel("Files")
        ax.set_title(key)
        ax.grid(True, alpha=0.3)

    axes[-1].hist(ok["tau_mean_amp_ns"].dropna(), bins=20,
                  color="#e74c3c", edgecolor="white", alpha=0.85)
    axes[-1].set_xlabel("τ_mean_amp (ns)")
    axes[-1].set_ylabel("Files")
    axes[-1].set_title("τ mean amp")
    axes[-1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(PTU_FOLDER / "batch_fit_tau_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nPlot saved → {PTU_FOLDER / 'batch_fit_tau_distributions.png'}")

── Summary statistics ───────────────────────────────────────
       tau1_ns  tau2_ns  tau3_ns  tau_mean_amp_ns  chi2_reduced_tail
count  32.0000  32.0000  32.0000          32.0000            32.0000
mean    6.2467   2.0367   0.4043           1.2492             1.8865
std     0.1530   0.0835   0.0254           0.0344             0.4463
min     5.9928   1.8588   0.3292           1.1590             1.5198
25%     6.1331   1.9887   0.3978           1.2360             1.6316
50%     6.2258   2.0344   0.4050           1.2456             1.7862
75%     6.3725   2.0940   0.4181           1.2662             1.9084
max     6.5292   2.2095   0.4516           1.3289             3.7110

Plot saved → /Volumes/Lexar/Layla/260309LMCR196TCAFADO40x.sptw/batch_fit_tau_distributions.png
